In [1]:
#1: IMPORTS 

from pypdf import PdfReader
import re, pickle, os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


#CELL 2: EXTRACT TEXT 

PDF_PATH = "tn_building_rules.pdf"   
reader = PdfReader(PDF_PATH)
pages_text = []
for page in reader.pages:
    t = page.extract_text()
    if t:
        pages_text.append(t)

full_text = "\n".join(pages_text)
print(f"Pages loaded : {len(pages_text)}")
print(f"Total chars  : {len(full_text):,}")
print("\nSample (first 300 chars):\n", full_text[:300])


#3: SMART CHUNKING FOR THIS DOCUMENT 
#
# This PDF uses three structural levels:
#   • PART –I / PART II / PART –III …       (major sections)
#   • 1. Short title … / 2. Definitions …   (individual rules)
#   • (1) "Access" means … / (2) …          (definition sub-items)
#
# Strategy:
#   1. Skip the Table of Contents (it ends before "RULES\nPART")
#   2. Split on numbered rules  e.g.  "27. Requirement for site…"
#   3. For Rule 2 (Definitions), further split on (1), (2), …
#   4. For long rules (>900 chars): sliding window with overlap
#   5. Attach the current PART label to every chunk for context

def extract_body(text):
    """Drop everything before 'RULES\nPART' to skip the TOC."""
    match = re.search(r'RULES\s*\nPART', text)
    if match:
        return text[match.start():]
    return text

def chunker(text, window=900, overlap=150):
    """Sliding-window split for long blocks."""
    chunks = []
    for start in range(0, len(text), window - overlap):
        piece = text[start:start + window].strip()
        if len(piece) > 80:
            chunks.append(piece)
    return chunks

body = extract_body(full_text)

# Split on numbered rules: "27." at line start (not inside sentences)
rule_pattern = re.compile(
    r'(?m)^(\d{1,3})\.\s+([A-Z][^\n]{5,})',  # "27. Requirement …"
)

# Collect PART labels as we scan
part_pattern = re.compile(r'PART\s*[–\-]?\s*([IVX]+)\b[^\n]*', re.IGNORECASE)

chunks = []
current_part = "General"

# Walk through the body and split by rule headers
positions = []
for m in rule_pattern.finditer(body):
    positions.append((m.start(), m.group(0), m.group(1)))

for i, (start, header, rule_num) in enumerate(positions):
    end = positions[i + 1][0] if i + 1 < len(positions) else len(body)
    block = body[start:end].strip()

    # Update current PART label if found in this block
    part_match = part_pattern.search(body[max(0, start - 200):start])
    if part_match:
        current_part = part_match.group(0).strip()

    prefix = f"[{current_part}] "

    # Rule 2 (Definitions): split on numbered sub-items (1), (2)…
    if rule_num == "2":
        sub_items = re.split(r'(?=\(\d+\)\s+")', block)
        for sub in sub_items:
            sub = sub.strip()
            if len(sub) > 60:
                chunks.append(prefix + sub)
    # Long rules: sliding window
    elif len(block) > 900:
        for piece in chunker(block, window=900, overlap=150):
            chunks.append(prefix + piece)
    # Short rules: keep whole
    elif len(block) > 60:
        chunks.append(prefix + block)

print(f"Total chunks : {len(chunks)}")
print("\nSample chunk:\n", chunks[10][:300])


# 4: EMBED + BUILD FAISS INDEX

INDEX_FILE  = "faiss_tn.bin"
CHUNKS_FILE = "chunks_tn.pkl"

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

if os.path.exists(INDEX_FILE) and os.path.exists(CHUNKS_FILE):
    print("Index already exists — loading saved index.")
    index = faiss.read_index(INDEX_FILE)
    with open(CHUNKS_FILE, "rb") as f:
        chunks = pickle.load(f)
else:
    print("Building embeddings …")
    embeddings = embed_model.encode(
        chunks, show_progress_bar=True, batch_size=64
    )
    embeddings = np.array(embeddings, dtype='float32')
    faiss.normalize_L2(embeddings)                 # cosine similarity

    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss.write_index(index, INDEX_FILE)
    with open(CHUNKS_FILE, "wb") as f:
        pickle.dump(chunks, f)
    print(f"✔ Index saved  ({len(chunks)} chunks, dim={dim})")


# 5: RETRIEVAL FUNCTION 
def retrieve(query: str, k: int = 5) -> list[str]:
    """
    Hybrid retrieval:
      • Keyword score  — exact term matches, length-normalised
      • Vector score   — cosine similarity via FAISS
      • Combined via Reciprocal Rank Fusion (RRF)
    """
    query_lower = query.lower()
    query_words = set(re.findall(r'\w+', query_lower))

    # Keyword scoring
    def kw_score(chunk):
        cl = chunk.lower()
        hits = sum(cl.count(w) for w in query_words)
        return hits / (1 + len(chunk) / 600)

    kw_ranked = sorted(range(len(chunks)),
                       key=lambda i: kw_score(chunks[i]), reverse=True)
    kw_rank   = {idx: r for r, idx in enumerate(kw_ranked)}

    # Vector scoring 
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype('float32')
    D, I  = index.search(q_vec, k * 4)
    vec_rank = {int(idx): r for r, idx in enumerate(I[0])}

    # RRF fusion 
    all_idx = set(list(vec_rank.keys()) + kw_ranked[:k * 4])
    def rrf(idx, c=60):
        return 1 / (c + kw_rank.get(idx, len(chunks))) + \
               1 / (c + vec_rank.get(idx, len(chunks)))

    top = sorted(all_idx, key=rrf, reverse=True)[:k]
    return [chunks[i] for i in top]


# 6: ASK FUNCTION (Jupyter-safe, TinyLlama) 

import ollama

def ask(query: str, k: int = 4, ctx_chars: int = 700, show_context: bool = True):
    """
    Query the RAG system.

    Parameters
    ----------
    query       : your question
    k           : number of chunks to retrieve
    ctx_chars   : characters to keep per chunk (lower = less RAM)
    show_context: print retrieved chunks for debugging
    """
    results   = retrieve(query, k=k)
    context   = "\n\n".join(
        f"[Chunk {i+1}]\n{c[:ctx_chars]}" for i, c in enumerate(results)
    )

    if show_context:
        print("─── Retrieved Chunks ───────────────────────────────")
        for i, c in enumerate(results):
            print(f"\n[{i+1}] {c[:200].strip()} …")
        print("─────────────────────────────────────────────────────\n")

    prompt = (
        "You are a legal assistant for Tamil Nadu building and development rules.\n"
        "Answer ONLY from the context below.\n"
        "If the answer is not in the context, say: 'Not found in the document.'\n"
        "Be concise. Mention the Rule number when it appears in context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\nAnswer:"
    )

    response = ollama.chat(
        model='phi',
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.1,   # factual / deterministic
            "num_predict": 300,   # short answer → less RAM pressure
        }
    )
    answer = response['message']['content']
    return answer


# 7: INTERACTIVE Q&A LOOP

print("TN Rules RAG — ready. Type your question below.\n")

while True:
    try:
        q = input("Question: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("Stopped.")
        break
    if not q:
        continue
    if q.lower() in ("exit", "quit", "q"):
        break
    print()
    ans = ask(q, show_context=True)
    print("Answer:\n", ans)
    print("\n" + "="*60 + "\n")
    

C:\Program Files\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pages loaded : 248
Total chars  : 509,151

Sample (first 300 chars):
 Tamil Nadu Combined Development and 
Building Rules, 2019  
 
 
 
 
 
 
 
February 2019 
 
 
 
Tamil Nadu Combined Development and 
Building Rules, 2019  
 
 
 
 
 
 
 
February 2019 
 
 
 
CONTENTS 
Rule 
No. Description Page 
No. 
 G.O.(Ms) No.18, Municipal Administration And Water Supply (MA.I) 

Total chunks : 703

Sample chunk:
 [PART – II] companied by proof of ownership, 
detailed plans, specifications, site plan, key plan and topo plan showing existing 
developments to a radius of 100 metres drawn to a scale of 1:500 and such other 
details as may be required from time to time shall be submitted to the competent 
authori


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4182.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index already exists — loading saved index.
TN Rules RAG — ready. Type your question below.



Question:  what is fsi?



─── Retrieved Chunks ───────────────────────────────

[1] [PART VI] 49. Premium FSI.— The Premium FSI over and above the normally allowable FSI relating 
the same to road width parameter may be allowed as follows: 
Sl.No. Road Width Premium FSI  
(% of norma …

[2] [PART VII] 4.  FSI Tolerance Limit FSI Tolerance limit will be 0.03 of FSI or 50 Sq.mt., 
floor area whichever is higher over and above the 
permissible FSI. …

[3] [PART VII] 4.  FSI Tolerance Limit FSI Tolerance limit will be 0.03 of FSI or 50 Sq.mt. 
floor area whichever is higher over and above the 
permissible FSI. …

[4] [PART VI] ording to the development. 
(b) Premium FSI charges shall not be levied for additional FSI upto 0.5 for High 
Rise Developments.  
(c) Premium FSI charges are applicable for Premium FSI achi …
─────────────────────────────────────────────────────

Answer:
  FSI stands for Floor Space Index. It's a measure used to determine the amount of usable space within a building relative to its total ar

Question:  q
